In [4]:
# imports and configuration
import gzip
import xml.etree.ElementTree as ET
import os
from collections import defaultdict, Counter
import pandas as pd
import random
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans

random.seed(42)

PUBMED_DIR = r"C:\Users\hp\Desktop\pubmed_raw"
MESH_OUTPUT = r"C:\Users\hp\Desktop\Research-System\training\datasets\mesh_counts.csv" # save all different MeSH terms for inspection
CLUSTER_OUTPUT = r"C:\Users\hp\Desktop\Research-System\training\datasets\mesh_clustered.csv" # save clustered MeSH terms for inspection
PUBMED_OUTPUT = r"C:\Users\hp\Desktop\Research-System\training\datasets\dataset_pubmed.csv"

CAP_PER_SUB = 20000

# priority order for tiebreaking
PRIORITY_ORDER = [
    "Oncology",
    "Neuroscience",
    "Genetics & Molecular Biology",
    "Immunology & Microbiology",
    "Cardiology",
    "Pharmacology"
]

# publication types to exclude: these are noise articles with no real abstracts
EXCLUDE_PUB_TYPES = {
    'Case Reports', 'Letter', 'Editorial', 'Comment',
    'Biography', 'Historical Article', 'News', 'Directory',
    'Portrait', 'Interview', 'Bibliography', 'Overall'
}

In [5]:
# pick the first file in the directory
first_file = sorted(os.listdir(PUBMED_DIR))[0]
file_path = os.path.join(PUBMED_DIR, first_file)

print(f"Reading: {first_file}\n")

with gzip.open(file_path, 'rb') as f:
    tree = ET.parse(f)
    root = tree.getroot()

# each paper is a PubmedArticle element
articles = root.findall('PubmedArticle')
print(f"Articles in this file: {len(articles):,}\n")

# find the first article that has a title, abstract, and MeSH terms
sample = None
for article in articles:
    title = article.findtext('.//ArticleTitle')
    abstract = article.findtext('.//AbstractText')
    mesh = article.findall('.//MeshHeading')
    if title and abstract and mesh:
        sample = article
        break

if sample is None:
    print("No article with all three fields found in this file.")
else:
    print("=" * 60)
    print("FULL STRUCTURE OF ONE PAPER")
    print("=" * 60)

    # PMID
    pmid = sample.findtext('.//PMID')
    print(f"\nPMID: {pmid}")

    # Title
    title = sample.findtext('.//ArticleTitle')
    print(f"\nTitle:\n  {title}")

    # Abstract can be structured (multiple AbstractText elements with labels)
    abstract_els = sample.findall('.//AbstractText')
    print(f"\nAbstract ({len(abstract_els)} part(s)):")
    for el in abstract_els:
        label = el.get('Label', 'UNLABELED')
        text = el.text or ''
        print(f"  [{label}]: {text[:200]}")

    # Publication type
    pub_types = [pt.text for pt in sample.findall('.//PublicationType')]
    print(f"\nPublication Types: {pub_types}")

    # Journal
    journal = sample.findtext('.//Journal/Title')
    print(f"\nJournal: {journal}")

    # Year
    year = sample.findtext('.//PubDate/Year')
    print(f"\nYear: {year}")

    # MeSH headings each has a DescriptorName and optional QualifierNames
    mesh_headings = sample.findall('.//MeshHeading')
    print(f"\nMeSH Headings ({len(mesh_headings)} total):")
    for mh in mesh_headings:
        descriptor = mh.find('DescriptorName')
        qualifiers = mh.findall('QualifierName')
        ui = descriptor.get('UI') if descriptor is not None else 'N/A'
        name = descriptor.text if descriptor is not None else 'N/A'
        major = descriptor.get('MajorTopicYN', 'N') if descriptor is not None else 'N'
        qual_names = [q.text for q in qualifiers]
        print(f"  [{ui}] {name} (Major: {major})")
        if qual_names:
            print(f"    Qualifiers: {qual_names}")

    # Keywords (separate from MeSH, author-provided)
    keywords = [k.text for k in sample.findall('.//Keyword')]
    print(f"\nKeywords: {keywords[:10]}")

Reading: pubmed26n0001.xml.gz

Articles in this file: 30,000

FULL STRUCTURE OF ONE PAPER

PMID: 21

Title:
  [Biochemical studies on camomile components/III. In vitro studies about the antipeptic activity of (--)-alpha-bisabolol (author's transl)].

Abstract (1 part(s)):
  [UNLABELED]: (--)-alpha-Bisabolol has a primary antipeptic action depending on dosage, which is not caused by an alteration of the pH-value. The proteolytic activity of pepsin is reduced by 50 percent through addi

Publication Types: ['English Abstract', 'Journal Article']

Journal: Arzneimittel-Forschung

Year: 1975

MeSH Headings (11 total):
  [D004305] Dose-Response Relationship, Drug (Major: N)
  [D006454] Hemoglobins (Major: N)
    Qualifiers: ['metabolism']
  [D006863] Hydrogen-Ion Concentration (Major: N)
  [D066298] In Vitro Techniques (Major: N)
  [D008722] Methods (Major: N)
  [D010434] Pepsin A (Major: N)
    Qualifiers: ['antagonists & inhibitors', 'metabolism']
  [D010946] Plants, Medicinal (Major: N)
 

In [6]:
# DATA QUALITY SCAN ACROSS ALL 50 FILES
# Before building any mapping we need to know:
# 1. How many papers are actually usable (have title + abstract + MeSH)
# 2. How common structured abstracts are (multiple labeled parts we need to join)
# 3. What publication types exist (so we can decide what to filter out)

# iterparse is used instead of ET.parse() because ET.parse() loads the entire
# XML file into memory at once before you can do anything with it.
# iterparse fires an event for each element as it is encountered in the file,
# so only one article is in memory at a time — much faster and lighter.

files = sorted(os.listdir(PUBMED_DIR))
print(f"Total files found: {len(files)}\n")

total_articles = 0
has_title = 0
has_abstract = 0
has_mesh = 0
has_all_three = 0
structured_abstracts = 0
pub_type_counts = Counter()
articles_per_file = []

for filename in files:
    file_path = os.path.join(PUBMED_DIR, filename)

    file_article_count = 0

    try:
        # gzip.open decompresses the .gz file on the fly as iterparse reads it
        # we never decompress the whole file to disk so it streams directly
        with gzip.open(file_path, 'rb') as f:

            # iterparse reads the XML stream and fires events as it goes
            # 'end' means the event fires when an element's closing tag is reached
            # so when we see event='end' and elem.tag='PubmedArticle', the entire
            # article element including all its children is fully loaded and ready
            for event, elem in ET.iterparse(f, events=('end',)):

                # we only care about complete PubmedArticle elements
                # all other elements (Journal, MeshHeading, etc.) are skipped here
                # because they are children of PubmedArticle and already inside it
                if elem.tag != 'PubmedArticle':
                    continue

                file_article_count += 1
                total_articles += 1

                # TITLE CHECK
                # findtext returns the text of the first matching element, or None
                # ArticleTitle exists in almost every record but can be an empty tag
                # so we strip whitespace and check it's actually non-empty
                title = elem.findtext('.//ArticleTitle')
                title_ok = bool(title and title.strip())

                # ABSTRACT CHECK
                # abstracts come in two formats:
                # FORMAT 1 — simple: one <AbstractText> element with plain text
                # FORMAT 2 — structured: multiple <AbstractText> elements each with
                #            a Label attribute like BACKGROUND, METHODS, RESULTS
                # we handle both by finding ALL AbstractText elements and joining them
                # with a space, which works correctly for both formats
                abstract_parts = elem.findall('.//AbstractText')
                abstract_text = ' '.join(
                    (part.text or '') for part in abstract_parts
                    # part.text is None if the element is empty, so we default to ''
                ).strip()
                abstract_ok = bool(abstract_text)

                # count how many papers use the structured abstract format
                # so we know our joining logic is actually being exercised
                if len(abstract_parts) > 1:
                    # more than one AbstractText element = definitely structured
                    structured_abstracts += 1
                elif len(abstract_parts) == 1 and abstract_parts[0].get('Label'):
                    # one AbstractText element but it has a Label attribute
                    # this is a structured abstract with only one section
                    structured_abstracts += 1

                # MESH CHECK
                # MeshHeading elements only exist if the article has been indexed
                # older articles, preprints, and some article types are not indexed
                # findall returns an empty list if none exist, so bool([]) = False
                mesh = elem.findall('.//MeshHeading')
                mesh_ok = bool(mesh)

                # TALLY
                if title_ok:
                    has_title += 1
                if abstract_ok:
                    has_abstract += 1
                if mesh_ok:
                    has_mesh += 1
                if title_ok and abstract_ok and mesh_ok:
                    # this is the count that actually matters — papers we can use
                    has_all_three += 1

                # PUBLICATION TYPES
                # each article can have multiple publication types
                # e.g. a paper can be both a Journal Article and a Review
                # we count all of them so we can see the full landscape
                # and decide later which types to keep and which to filter out
                for pt in elem.findall('.//PublicationType'):
                    if pt.text:
                        pub_type_counts[pt.text] += 1

                # MEMORY CLEANUP
                # this makes iterparse memory-efficient
                # after we are done processing this article we tell Python to
                # discard the element and all its children from memory
                # without this line every article would accumulate in RAM
                # and after 1.5 million articles the process would crash
                elem.clear()

    except (EOFError, ET.ParseError) as e:
        # EOFError means the .gz file was cut off mid-download
        # ParseError means the XML inside is malformed
        # either way we skip incomplete files, keep whatever articles we already counted,
        # and continue to the next file
        print(f"  SKIPPED {filename}: {e}")

    articles_per_file.append(file_article_count)
    print(f"  {filename}: {file_article_count:,} articles")

print("\n" + "=" * 60)
print("OVERALL COUNTS")
print("=" * 60)
print(f"Total articles scanned:          {total_articles:>10,}")
print(f"Have title:                      {has_title:>10,}")
print(f"Have abstract:                   {has_abstract:>10,}")
print(f"Have MeSH terms:                 {has_mesh:>10,}")
print(f"Have all three (usable):         {has_all_three:>10,}")
print(f"Usable %:                        {has_all_three/total_articles*100:>9.1f}%")
print(f"Structured abstracts:            {structured_abstracts:>10,}")

print("\n" + "=" * 60)
print("ARTICLES PER FILE")
print("=" * 60)
print(f"Min: {min(articles_per_file):,}")
print(f"Max: {max(articles_per_file):,}")
print(f"Avg: {sum(articles_per_file)//len(articles_per_file):,}")

print("\n" + "=" * 60)
print("TOP 30 PUBLICATION TYPES")
print("=" * 60)
for pub_type, count in pub_type_counts.most_common(30):
    print(f"  {pub_type:<45} {count:>10,}")

Total files found: 50

  pubmed26n0001.xml.gz: 30,000 articles
  pubmed26n0002.xml.gz: 30,000 articles
  pubmed26n0003.xml.gz: 30,000 articles
  pubmed26n0004.xml.gz: 30,000 articles
  pubmed26n0005.xml.gz: 30,000 articles
  pubmed26n0006.xml.gz: 30,000 articles
  pubmed26n0007.xml.gz: 30,000 articles
  pubmed26n0008.xml.gz: 30,000 articles
  pubmed26n0009.xml.gz: 30,000 articles
  pubmed26n0010.xml.gz: 30,000 articles
  pubmed26n0011.xml.gz: 30,000 articles
  pubmed26n0012.xml.gz: 30,000 articles
  pubmed26n0013.xml.gz: 30,000 articles
  pubmed26n0014.xml.gz: 30,000 articles
  pubmed26n0015.xml.gz: 30,000 articles
  pubmed26n0016.xml.gz: 30,000 articles
  pubmed26n0017.xml.gz: 30,000 articles
  pubmed26n0018.xml.gz: 30,000 articles
  pubmed26n0019.xml.gz: 30,000 articles
  pubmed26n0020.xml.gz: 30,000 articles
  pubmed26n0021.xml.gz: 30,000 articles
  pubmed26n0022.xml.gz: 30,000 articles
  pubmed26n0023.xml.gz: 30,000 articles
  pubmed26n0024.xml.gz: 30,000 articles
  pubmed26n0025.x

In [7]:
# MESH TERM DUMP ACROSS ALL 50 FILES
# Scan every article that is usable (has title + abstract + MeSH)
# and collect every unique MeSH descriptor name, its UI number,
# and how many papers use it.
# The output is saved to mesh_counts.csv so we can open it in Excel,
# sort by count, and manually decide which terms map to which subcategory.

import pandas as pd

files = sorted(os.listdir(PUBMED_DIR))

# mesh_term_counts maps (ui, descriptor_name) -> paper count
# using a tuple key so we keep the UI number and name together
mesh_term_counts = Counter()

total_scanned = 0
total_usable = 0
total_excluded_by_type = 0

for filename in files:
    file_path = os.path.join(PUBMED_DIR, filename)

    try:
        with gzip.open(file_path, 'rb') as f:
            for event, elem in ET.iterparse(f, events=('end',)):
                if elem.tag != 'PubmedArticle':
                    continue

                total_scanned += 1

                # check title
                title = elem.findtext('.//ArticleTitle')
                title_ok = bool(title and title.strip())

                # check abstract, join all parts
                abstract_parts = elem.findall('.//AbstractText')
                abstract_text = ' '.join(
                    (part.text or '') for part in abstract_parts
                ).strip()
                abstract_ok = bool(abstract_text)

                # check MeSH
                mesh_headings = elem.findall('.//MeshHeading')
                mesh_ok = bool(mesh_headings)

                # skip if missing any required field
                if not (title_ok and abstract_ok and mesh_ok):
                    elem.clear()
                    continue

                # check publication types, collect all types for this article
                pub_types = set()
                for pt in elem.findall('.//PublicationType'):
                    if pt.text:
                        pub_types.add(pt.text)

                # skip if any of its publication types are in the exclusion list
                # we use intersection — if the sets share any element, skip
                if pub_types & EXCLUDE_PUB_TYPES:
                    total_excluded_by_type += 1
                    elem.clear()
                    continue

                total_usable += 1

                # collect every MeSH descriptor on this article
                # each MeshHeading has one DescriptorName element
                # UI is the unique identifier e.g. D006153
                # the text is the human-readable name e.g. "Genomics"
                for mh in mesh_headings:
                    descriptor = mh.find('DescriptorName')
                    if descriptor is not None and descriptor.text:
                        ui = descriptor.get('UI', 'UNKNOWN')
                        name = descriptor.text.strip()
                        mesh_term_counts[(ui, name)] += 1

                elem.clear()

    except (EOFError, ET.ParseError) as e:
        print(f"  SKIPPED {filename}: {e}")

print("=" * 60)
print("SCAN COMPLETE")
print("=" * 60)
print(f"Total articles scanned:          {total_scanned:>10,}")
print(f"Excluded by publication type:    {total_excluded_by_type:>10,}")
print(f"Usable articles:                 {total_usable:>10,}")
print(f"Unique MeSH terms found:         {len(mesh_term_counts):>10,}")

# convert to dataframe and save
# columns: ui, descriptor_name, paper_count
rows = [
    {'ui': ui, 'descriptor_name': name, 'paper_count': count}
    for (ui, name), count in mesh_term_counts.items()
]
df_mesh = pd.DataFrame(rows)

# sort by paper count descending so the most common terms are at the top
df_mesh = df_mesh.sort_values('paper_count', ascending=False).reset_index(drop=True)

df_mesh.to_csv(MESH_OUTPUT, index=False)
print(f"\nSaved to {MESH_OUTPUT}")

print("\n" + "=" * 60)
print("TOP 50 MESH TERMS BY PAPER COUNT")
print("=" * 60)
print(f"{'Rank':<6} {'UI':<12} {'Count':>8}  {'Descriptor Name'}")
print("-" * 60)
for i, row in df_mesh.head(50).iterrows():
    print(f"  {i+1:<4} {row['ui']:<12} {row['paper_count']:>8,}  {row['descriptor_name']}")

  SKIPPED pubmed26n0045.xml.gz: Compressed file ended before the end-of-stream marker was reached
  SKIPPED pubmed26n0046.xml.gz: Compressed file ended before the end-of-stream marker was reached
SCAN COMPLETE
Total articles scanned:           1,467,775
Excluded by publication type:        69,637
Usable articles:                    616,309
Unique MeSH terms found:             18,615

Saved to C:\Users\hp\Desktop\Research-System\training\datasets\mesh_counts.csv

TOP 50 MESH TERMS BY PAPER COUNT
Rank   UI              Count  Descriptor Name
------------------------------------------------------------
  1    D006801       328,318  Humans
  2    D000818       248,837  Animals
  3    D008297       214,591  Male
  4    D005260       202,224  Female
  5    D000328       117,172  Adult
  6    D008875        90,920  Middle Aged
  7    D051381        78,224  Rats
  8    D000368        58,982  Aged
  9    D013997        54,268  Time Factors
  10   D000293        53,186  Adolescent
  11   D051379

In [8]:
# MESH TERM CLUSTERING USING SENTENCE EMBEDDINGS + KMEANS
# We embed each MeSH descriptor name into a semantic vector,
# then cluster those vectors into 25 groups.
# 25 clusters gives tight meaningful groups that we can then
# manually merge into our 6 Biology & Medicine subcategories.

# load the mesh counts CSV and filter to terms with 500+ papers
df_mesh = pd.read_csv(MESH_OUTPUT)
df_mesh = df_mesh[df_mesh['paper_count'] >= 500].reset_index(drop=True)
print(f"Terms with 500+ papers: {len(df_mesh):,}")

# load the same embedding model used in the recommender
# it converts each term name into a 384-dimensional vector
# terms with similar meanings will have similar vectors
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# embed all descriptor names
# convert_to_numpy=True gives us a numpy array that sklearn can work with
print("Embedding MeSH terms...")
term_names = df_mesh['descriptor_name'].tolist()
embeddings = embedding_model.encode(term_names, convert_to_numpy=True, show_progress_bar=True)
print(f"Embeddings shape: {embeddings.shape}")

# run KMeans with 25 clusters
# n_init=10 means KMeans tries 10 different random starting points
# and keeps the best result, reduces chance of bad clustering
print("Running KMeans...")
N_CLUSTERS = 25
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(embeddings)

# attach cluster labels back to the dataframe
df_mesh['cluster'] = cluster_labels

df_mesh.to_csv(CLUSTER_OUTPUT, index=False)
print(f"\nSaved to {CLUSTER_OUTPUT}")

# print each cluster with its top 20 terms by paper count
# this gives us a quick overview of what each cluster contains
print("\n" + "=" * 60)
print("CLUSTER CONTENTS (top 20 terms each, sorted by count)")
print("=" * 60)
for cluster_id in range(N_CLUSTERS):
    cluster_df = df_mesh[df_mesh['cluster'] == cluster_id].sort_values(
        'paper_count', ascending=False
    )
    print(f"\n--- Cluster {cluster_id} ({len(cluster_df)} terms) ---")
    for _, row in cluster_df.head(20).iterrows():
        print(f"  {row['paper_count']:>8,}  {row['descriptor_name']}")

Terms with 500+ papers: 2,318


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2286.92it/s]


Embedding MeSH terms...


Batches: 100%|██████████| 73/73 [00:09<00:00,  8.10it/s]


Embeddings shape: (2318, 384)
Running KMeans...

Saved to C:\Users\hp\Desktop\Research-System\training\datasets\mesh_clustered.csv

CLUSTER CONTENTS (top 20 terms each, sorted by count)

--- Cluster 0 (89 terms) ---
    16,003  Cells, Cultured
    12,922  Cell Line
    11,238  Cell Membrane
     9,961  Erythrocytes
     9,183  Lymphocytes
     8,352  Cell Division
     6,945  Cell Nucleus
     5,207  Epithelium
     5,016  Histocytochemistry
     4,783  Macrophages
     4,700  Membrane Potentials
     4,518  Biological Transport
     4,488  Cell Differentiation
     4,404  Fibroblasts
     4,164  Cytoplasm
     4,133  Mitochondria
     4,042  Leukocytes
     3,803  Cell Survival
     3,738  Tumor Cells, Cultured
     3,474  Neutrophils

--- Cluster 1 (89 terms) ---
     8,471  Prognosis
     6,237  Neoplasms, Experimental
     5,483  Neoplasms
     4,854  Neoplasm Metastasis
     4,714  Breast Neoplasms
     4,219  Lung Neoplasms
     4,183  Adenocarcinoma
     3,696  Liver Neoplasms
 

In [9]:
# MESH MAPPING
PUBMED_MESH_MAP = {

    # NEUROSCIENCE weight 1.0
    "Brain":                                {"sub": "Neuroscience", "weight": 1.0},
    "Neurons":                              {"sub": "Neuroscience", "weight": 1.0},
    "Cerebral Cortex":                      {"sub": "Neuroscience", "weight": 1.0},
    "Electroencephalography":               {"sub": "Neuroscience", "weight": 1.0},
    "Evoked Potentials":                    {"sub": "Neuroscience", "weight": 1.0},
    "Electrophysiology":                    {"sub": "Neuroscience", "weight": 1.0},
    "Spinal Cord":                          {"sub": "Neuroscience", "weight": 1.0},
    "Axons":                                {"sub": "Neuroscience", "weight": 1.0},
    "Brain Chemistry":                      {"sub": "Neuroscience", "weight": 1.0},
    "Synapses":                             {"sub": "Neuroscience", "weight": 1.0},
    "Hippocampus":                          {"sub": "Neuroscience", "weight": 1.0},
    "Electromyography":                     {"sub": "Neuroscience", "weight": 1.0},
    "Brain Mapping":                        {"sub": "Neuroscience", "weight": 1.0},
    "Synaptic Transmission":                {"sub": "Neuroscience", "weight": 1.0},
    "Sympathetic Nervous System":           {"sub": "Neuroscience", "weight": 1.0},
    "Neural Pathways":                      {"sub": "Neuroscience", "weight": 1.0},
    "Motor Neurons":                        {"sub": "Neuroscience", "weight": 1.0},
    "Seizures":                             {"sub": "Neuroscience", "weight": 1.0},
    "Nerve Tissue Proteins":                {"sub": "Neuroscience", "weight": 1.0},
    "Cerebellum":                           {"sub": "Neuroscience", "weight": 1.0},
    "Neural Conduction":                    {"sub": "Neuroscience", "weight": 1.0},
    "Brain Stem":                           {"sub": "Neuroscience", "weight": 1.0},
    "Epilepsy":                             {"sub": "Neuroscience", "weight": 1.0},
    "Corpus Striatum":                      {"sub": "Neuroscience", "weight": 1.0},
    "Neuromuscular Junction":               {"sub": "Neuroscience", "weight": 1.0},
    "Visual Cortex":                        {"sub": "Neuroscience", "weight": 1.0},
    "Central Nervous System":               {"sub": "Neuroscience", "weight": 1.0},
    "Neuroglia":                            {"sub": "Neuroscience", "weight": 1.0},
    "Neurons Afferent":                     {"sub": "Neuroscience", "weight": 1.0},
    "Neural Inhibition":                    {"sub": "Neuroscience", "weight": 1.0},
    "Peripheral Nerves":                    {"sub": "Neuroscience", "weight": 1.0},
    "Nerve Fibers":                         {"sub": "Neuroscience", "weight": 1.0},
    "Mesencephalon":                        {"sub": "Neuroscience", "weight": 1.0},
    "Nerve Degeneration":                   {"sub": "Neuroscience", "weight": 1.0},
    "Brain Injuries":                       {"sub": "Neuroscience", "weight": 1.0},
    "Medulla Oblongata":                    {"sub": "Neuroscience", "weight": 1.0},
    "Afferent Pathways":                    {"sub": "Neuroscience", "weight": 1.0},
    "Parkinson Disease":                    {"sub": "Neuroscience", "weight": 1.0},
    "Synaptosomes":                         {"sub": "Neuroscience", "weight": 1.0},
    "Blood-Brain Barrier":                  {"sub": "Neuroscience", "weight": 1.0},
    "Sciatic Nerve":                        {"sub": "Neuroscience", "weight": 1.0},
    "Visual Pathways":                      {"sub": "Neuroscience", "weight": 1.0},
    "Myelin Sheath":                        {"sub": "Neuroscience", "weight": 1.0},
    "Cerebrospinal Fluid":                  {"sub": "Neuroscience", "weight": 1.0},
    "Autonomic Nervous System":             {"sub": "Neuroscience", "weight": 1.0},
    "Spinal Cord Injuries":                 {"sub": "Neuroscience", "weight": 1.0},
    "Nerve Endings":                        {"sub": "Neuroscience", "weight": 1.0},
    "Optic Nerve":                          {"sub": "Neuroscience", "weight": 1.0},
    "Nerve Regeneration":                   {"sub": "Neuroscience", "weight": 1.0},
    "Somatosensory Cortex":                 {"sub": "Neuroscience", "weight": 1.0},
    "Motor Cortex":                         {"sub": "Neuroscience", "weight": 1.0},
    "Frontal Lobe":                         {"sub": "Neuroscience", "weight": 1.0},
    "Nervous System":                       {"sub": "Neuroscience", "weight": 1.0},
    "Ganglia Spinal":                       {"sub": "Neuroscience", "weight": 1.0},
    "Peripheral Nervous System Diseases":   {"sub": "Neuroscience", "weight": 1.0},
    "Synaptic Membranes":                   {"sub": "Neuroscience", "weight": 1.0},
    "Nerve Fibers Myelinated":              {"sub": "Neuroscience", "weight": 1.0},
    "Spinal Nerve Roots":                   {"sub": "Neuroscience", "weight": 1.0},
    "Brain Damage Chronic":                 {"sub": "Neuroscience", "weight": 1.0},
    "Brain Ischemia":                       {"sub": "Neuroscience", "weight": 1.0},
    "Synaptic Vesicles":                    {"sub": "Neuroscience", "weight": 1.0},
    "Limbic System":                        {"sub": "Neuroscience", "weight": 1.0},
    "Nerve Growth Factors":                 {"sub": "Neuroscience", "weight": 1.0},
    "Alzheimer Disease":                    {"sub": "Neuroscience", "weight": 1.0},
    "Dementia":                             {"sub": "Neuroscience", "weight": 1.0},
    "Multiple Sclerosis":                   {"sub": "Neuroscience", "weight": 1.0},
    "Brain Diseases":                       {"sub": "Neuroscience", "weight": 1.0},
    "Nervous System Diseases":              {"sub": "Neuroscience", "weight": 1.0},
    "Central Nervous System Diseases":      {"sub": "Neuroscience", "weight": 1.0},

    # NEUROSCIENCE weight 0.5
    "Electric Stimulation":                 {"sub": "Neuroscience", "weight": 0.5},
    "Acoustic Stimulation":                 {"sub": "Neuroscience", "weight": 0.5},
    "Photic Stimulation":                   {"sub": "Neuroscience", "weight": 0.5},
    "Intellectual Disability":              {"sub": "Neuroscience", "weight": 0.5},
    "Neurotic Disorders":                   {"sub": "Neuroscience", "weight": 0.5},
    
    # CARDIOLOGY weight 1.0
    "Heart Rate":                           {"sub": "Cardiology", "weight": 1.0},
    "Myocardium":                           {"sub": "Cardiology", "weight": 1.0},
    "Hypertension":                         {"sub": "Cardiology", "weight": 1.0},
    "Heart":                                {"sub": "Cardiology", "weight": 1.0},
    "Hemodynamics":                         {"sub": "Cardiology", "weight": 1.0},
    "Electrocardiography":                  {"sub": "Cardiology", "weight": 1.0},
    "Coronary Disease":                     {"sub": "Cardiology", "weight": 1.0},
    "Myocardial Infarction":                {"sub": "Cardiology", "weight": 1.0},
    "Myocardial Contraction":               {"sub": "Cardiology", "weight": 1.0},
    "Cardiac Output":                       {"sub": "Cardiology", "weight": 1.0},
    "Heart Ventricles":                     {"sub": "Cardiology", "weight": 1.0},
    "Vascular Resistance":                  {"sub": "Cardiology", "weight": 1.0},
    "Aorta":                                {"sub": "Cardiology", "weight": 1.0},
    "Arrhythmias Cardiac":                  {"sub": "Cardiology", "weight": 1.0},
    "Arteriosclerosis":                     {"sub": "Cardiology", "weight": 1.0},
    "Coronary Circulation":                 {"sub": "Cardiology", "weight": 1.0},
    "Angiography":                          {"sub": "Cardiology", "weight": 1.0},
    "Coronary Vessels":                     {"sub": "Cardiology", "weight": 1.0},
    "Heart Failure":                        {"sub": "Cardiology", "weight": 1.0},
    "Echocardiography":                     {"sub": "Cardiology", "weight": 1.0},
    "Heart Conduction System":              {"sub": "Cardiology", "weight": 1.0},
    "Cardiac Catheterization":              {"sub": "Cardiology", "weight": 1.0},
    "Heart Diseases":                       {"sub": "Cardiology", "weight": 1.0},
    "Angina Pectoris":                      {"sub": "Cardiology", "weight": 1.0},
    "Heart Atria":                          {"sub": "Cardiology", "weight": 1.0},
    "Pulmonary Artery":                     {"sub": "Cardiology", "weight": 1.0},
    "Heart Defects Congenital":             {"sub": "Cardiology", "weight": 1.0},
    "Coronary Artery Bypass":               {"sub": "Cardiology", "weight": 1.0},
    "Heart Valve Prosthesis":               {"sub": "Cardiology", "weight": 1.0},
    "Cardiomegaly":                         {"sub": "Cardiology", "weight": 1.0},
    "Arterial Occlusive Diseases":          {"sub": "Cardiology", "weight": 1.0},
    "Carotid Arteries":                     {"sub": "Cardiology", "weight": 1.0},
    "Muscle Smooth Vascular":               {"sub": "Cardiology", "weight": 1.0},
    "Coronary Angiography":                 {"sub": "Cardiology", "weight": 1.0},
    "Cardiovascular Diseases":              {"sub": "Cardiology", "weight": 1.0},
    "Mitral Valve":                         {"sub": "Cardiology", "weight": 1.0},
    "Pacemaker Artificial":                 {"sub": "Cardiology", "weight": 1.0},
    "Heart Block":                          {"sub": "Cardiology", "weight": 1.0},
    "Cardiac Surgical Procedures":          {"sub": "Cardiology", "weight": 1.0},
    "Pulmonary Embolism":                   {"sub": "Cardiology", "weight": 1.0},
    "Aortic Valve":                         {"sub": "Cardiology", "weight": 1.0},
    "Tachycardia":                          {"sub": "Cardiology", "weight": 1.0},
    "Vasoconstriction":                     {"sub": "Cardiology", "weight": 1.0},
    "Ischemic Attack Transient":            {"sub": "Cardiology", "weight": 1.0},
    "Aorta Abdominal":                      {"sub": "Cardiology", "weight": 1.0},
    "Angiocardiography":                    {"sub": "Cardiology", "weight": 1.0},
    "Heart Valve Diseases":                 {"sub": "Cardiology", "weight": 1.0},
    "Cardiopulmonary Bypass":               {"sub": "Cardiology", "weight": 1.0},
    "Vasodilation":                         {"sub": "Cardiology", "weight": 1.0},
    "Mitral Valve Insufficiency":           {"sub": "Cardiology", "weight": 1.0},
    "Cardiomyopathies":                     {"sub": "Cardiology", "weight": 1.0},
    "Anti-Arrhythmia Agents":               {"sub": "Cardiology", "weight": 1.0},
    "Extracorporeal Circulation":           {"sub": "Cardiology", "weight": 1.0},
    "Pulmonary Edema":                      {"sub": "Cardiology", "weight": 1.0},
    "Fetal Heart":                          {"sub": "Cardiology", "weight": 1.0},
    "Ventricular Function":                 {"sub": "Cardiology", "weight": 1.0},
    "Ventricular Fibrillation":             {"sub": "Cardiology", "weight": 1.0},
    "Cardiac Pacing Artificial":            {"sub": "Cardiology", "weight": 1.0},
    "Ventricular Function Left":            {"sub": "Cardiology", "weight": 1.0},
    "Aortic Valve Stenosis":                {"sub": "Cardiology", "weight": 1.0},
    "Intracranial Aneurysm":                {"sub": "Cardiology", "weight": 1.0},
    "Mitral Valve Stenosis":                {"sub": "Cardiology", "weight": 1.0},
    "Cardiovascular System":                {"sub": "Cardiology", "weight": 1.0},
    "Stroke Volume":                        {"sub": "Cardiology", "weight": 1.0},
    "Heart Septal Defects Ventricular":     {"sub": "Cardiology", "weight": 1.0},
    "Heart Transplantation":                {"sub": "Cardiology", "weight": 1.0},
    "Hypertension Pulmonary":               {"sub": "Cardiology", "weight": 1.0},
    "Disseminated Intravascular Coagulation": {"sub": "Cardiology", "weight": 1.0},
    "Cardiac Volume":                       {"sub": "Cardiology", "weight": 1.0},
    "Collateral Circulation":               {"sub": "Cardiology", "weight": 1.0},
    "Aorta Thoracic":                       {"sub": "Cardiology", "weight": 1.0},
    "Femoral Artery":                       {"sub": "Cardiology", "weight": 1.0},
    "Renal Artery":                         {"sub": "Cardiology", "weight": 1.0},
    "Mesenteric Arteries":                  {"sub": "Cardiology", "weight": 1.0},
    "Saphenous Vein":                       {"sub": "Cardiology", "weight": 1.0},
    "Thromboembolism":                      {"sub": "Cardiology", "weight": 1.0},
    "Pulmonary Circulation":                {"sub": "Cardiology", "weight": 1.0},
    "Blood Flow Velocity":                  {"sub": "Cardiology", "weight": 1.0},
    "Endothelium Vascular":                 {"sub": "Cardiology", "weight": 1.0},
    "Blood Vessels":                        {"sub": "Cardiology", "weight": 1.0},
    "Cerebral Angiography":                 {"sub": "Cardiology", "weight": 1.0},

    # CARDIOLOGY weight 0.5
    "Blood Pressure":                       {"sub": "Cardiology", "weight": 0.5},
    "Perfusion":                            {"sub": "Cardiology", "weight": 0.5},
    "Arteries":                             {"sub": "Cardiology", "weight": 0.5},
    "Capillaries":                          {"sub": "Cardiology", "weight": 0.5},
    "Ischemia":                             {"sub": "Cardiology", "weight": 0.5},
    "Hemoglobins":                          {"sub": "Cardiology", "weight": 0.5},
    "Blood Platelets":                      {"sub": "Cardiology", "weight": 0.5},
    "Regional Blood Flow":                  {"sub": "Cardiology", "weight": 0.5},
    "Thrombosis":                           {"sub": "Cardiology", "weight": 0.5},
    "Endothelium":                          {"sub": "Cardiology", "weight": 0.5},
    "Blood Coagulation":                    {"sub": "Cardiology", "weight": 0.5},
    "Thrombin":                             {"sub": "Cardiology", "weight": 0.5},
    "Veins":                                {"sub": "Cardiology", "weight": 0.5},
    "Angiotensin II":                       {"sub": "Cardiology", "weight": 0.5},
    
    # GENETICS & MOLECULAR BIOLOGY weight 1.0
    "Molecular Sequence Data":              {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "DNA":                                  {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Base Sequence":                        {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Amino Acid Sequence":                  {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Mutation":                             {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Protein Binding":                      {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "RNA Messenger":                        {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Protein Conformation":                 {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Proteins":                             {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Genes":                                {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Transcription Genetic":                {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Peptides":                             {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Protein Biosynthesis":                 {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "RNA":                                  {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Peptide Fragments":                    {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Chromosome Mapping":                   {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Cloning Molecular":                    {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Saccharomyces cerevisiae":             {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Polymerase Chain Reaction":            {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Nucleic Acid Hybridization":           {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Molecular Conformation":               {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Chromosome Aberrations":               {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "DNA Replication":                      {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Nucleic Acid Conformation":            {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Recombinant Proteins":                 {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Chromosomes":                          {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Gene Expression":                      {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Pedigree":                             {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Ribosomes":                            {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Genetic Variation":                    {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Genotype":                             {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Alleles":                              {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Recombination Genetic":                {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Genetic Linkage":                      {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Oligodeoxyribonucleotides":            {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Mutagens":                             {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Chromatin":                            {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Ribonucleases":                        {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Biological Evolution":                 {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Protein Kinases":                      {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Polymorphism Genetic":                 {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Gene Frequency":                       {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "RNA Transfer":                         {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "RNA Ribosomal":                        {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "DNA Repair":                           {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Heterozygote":                         {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Gene Expression Regulation":           {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "DNA Restriction Enzymes":              {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Sequence Homology Nucleic Acid":       {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "DNA-Directed RNA Polymerases":         {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Drosophila melanogaster":              {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Promoter Regions Genetic":             {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Crosses Genetic":                      {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Sequence Homology Amino Acid":         {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "DNA-Binding Proteins":                 {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Morphogenesis":                        {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "RNA Bacterial":                        {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Polymorphism, Restriction Fragment Length": {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Nucleic Acid Denaturation":            {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Protein Denaturation":                 {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "DNA Single-Stranded":                  {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Recombinant Fusion Proteins":          {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Protein Precursors":                   {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "DNA Probes":                           {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Adenine Nucleotides":                  {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Transcription Factors":                {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Templates Genetic":                    {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Sex Chromosomes":                      {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Protein Kinase C":                     {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Mutagenesis Site-Directed":            {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Homozygote":                           {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Genes Dominant":                       {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Transduction Genetic":                 {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Ribosomal Proteins":                   {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Hybridization Genetic":                {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Translocation Genetic":                {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Nucleoproteins":                       {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Meiosis":                              {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "DNA Recombinant":                      {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Genes Recessive":                      {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Conjugation Genetic":                  {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Genetic Complementation Test":         {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "GTP-Binding Proteins":                 {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "RNA-Directed DNA Polymerase":          {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Sequence Alignment":                   {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Selection Genetic":                    {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Oligonucleotides":                     {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Repetitive Sequences Nucleic Acid":    {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "DNA Circular":                         {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Transformation Genetic":               {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Genes Regulator":                      {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Drosophila":                           {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "DNA-Directed DNA Polymerase":          {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Genetic Markers":                      {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Nuclear Proteins":                     {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Nucleotides":                          {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Chromosome Deletion":                  {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Genetic Vectors":                      {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "DNA Transposable Elements":            {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Genetics Population":                  {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Extrachromosomal Inheritance":         {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "In Situ Hybridization":                {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "X Chromosome":                         {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "DNA Mitochondrial":                    {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Guanine Nucleotides":                  {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Chromosome Disorders":                 {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Protein-Tyrosine Kinases":             {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Multigene Family":                     {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "DNA Mutational Analysis":              {"sub": "Genetics & Molecular Biology", "weight": 1.0},
    "Oligoribonucleotides":                 {"sub": "Genetics & Molecular Biology", "weight": 1.0},

    # GENETICS & MOLECULAR BIOLOGY weight 0.5
    "Phenotype":                            {"sub": "Genetics & Molecular Biology", "weight": 0.5},
    "DNA Viral":                            {"sub": "Genetics & Molecular Biology", "weight": 0.5},
    "Microbiology":                         {"sub": "Genetics & Molecular Biology", "weight": 0.5},
    "Virus Replication":                    {"sub": "Genetics & Molecular Biology", "weight": 0.5},
    "Viral Proteins":                       {"sub": "Genetics & Molecular Biology", "weight": 0.5},
    "RNA Viral":                            {"sub": "Genetics & Molecular Biology", "weight": 0.5},
    "Genes Viral":                          {"sub": "Genetics & Molecular Biology", "weight": 0.5},
    "RNA Viruses":                          {"sub": "Genetics & Molecular Biology", "weight": 0.5},
    "Carrier Proteins":                     {"sub": "Genetics & Molecular Biology", "weight": 0.5},
    "Blood Proteins":                       {"sub": "Genetics & Molecular Biology", "weight": 0.5},
    "Muscle Proteins":                      {"sub": "Genetics & Molecular Biology", "weight": 0.5},
    "Plant Proteins":                       {"sub": "Genetics & Molecular Biology", "weight": 0.5},
    "Fungal Proteins":                      {"sub": "Genetics & Molecular Biology", "weight": 0.5},
    "Proto-Oncogene Proteins":              {"sub": "Genetics & Molecular Biology", "weight": 0.5},
    "Molecular Structure":                  {"sub": "Genetics & Molecular Biology", "weight": 0.5},
    
    # IMMUNOLOGY & MICROBIOLOGY weight 1.0
    "Escherichia coli":                     {"sub": "Immunology & Microbiology", "weight": 1.0},
    "T-Lymphocytes":                        {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Immunoglobulin G":                     {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Anti-Bacterial Agents":                {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Fluorescent Antibody Technique":       {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Radioimmunoassay":                     {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Lymphocyte Activation":                {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Antibody Formation":                   {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Antibodies Viral":                     {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Bacteria":                             {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Epitopes":                             {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Immunity Cellular":                    {"sub": "Immunology & Microbiology", "weight": 1.0},
    "B-Lymphocytes":                        {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Antibodies":                           {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Antigens":                             {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Immunoglobulin M":                     {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Immune Sera":                          {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Antigens Viral":                       {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Bacterial Proteins":                   {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Drug Resistance Microbial":            {"sub": "Immunology & Microbiology", "weight": 1.0},
    "DNA Bacterial":                        {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Immunodiffusion":                      {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Antibody Specificity":                 {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Microbial Sensitivity Tests":          {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Immunohistochemistry":                 {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Phagocytosis":                         {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Immunoglobulins":                      {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Antigens Bacterial":                   {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Complement System Proteins":           {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Immunization":                         {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Bacterial Infections":                 {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Antibodies Monoclonal":                {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Immunoglobulin A":                     {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Antibodies Bacterial":                 {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Cytotoxicity Tests Immunologic":       {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Coliphages":                           {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Staphylococcus aureus":                {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Antigen-Antibody Reactions":           {"sub": "Immunology & Microbiology", "weight": 1.0},
    "HLA Antigens":                         {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Immunoenzyme Techniques":              {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Enzyme-Linked Immunosorbent Assay":    {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Immunoelectrophoresis":                {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Histocompatibility Antigens":          {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Antigen-Antibody Complex":             {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Binding Sites Antibody":               {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Autoantibodies":                       {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Pseudomonas aeruginosa":               {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Salmonella typhimurium":               {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Bacteriological Techniques":           {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Vaccination":                          {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Simplexvirus":                         {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Hypersensitivity Delayed":             {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Cytotoxicity Immunologic":             {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Immunity":                             {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Immunoglobulin E":                     {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Anaerobiosis":                         {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Hepatitis B":                          {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Lupus Erythematosus Systemic":         {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Polysaccharides Bacterial":            {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Bacillus subtilis":                    {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Streptococcus":                        {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Simian virus 40":                      {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Staphylococcus":                       {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Water Microbiology":                   {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Staphylococcal Infections":            {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Immunologic Techniques":               {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Virulence":                            {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Dose-Response Relationship Immunologic": {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Pseudomonas":                          {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Lymphocyte Culture Test Mixed":        {"sub": "Immunology & Microbiology", "weight": 1.0},
    "BCG Vaccine":                          {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Virus Diseases":                       {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Antilymphocyte Serum":                 {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Enterobacteriaceae":                   {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Receptors Antigen B-Cell":             {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Fungi":                                {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Cephalosporins":                       {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Immunization Passive":                 {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Influenza A virus":                    {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Salmonella":                           {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Herpesvirus 4 Human":                  {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Autoimmune Diseases":                  {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Immunotherapy":                        {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Tumor Necrosis Factor-alpha":          {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Immune Tolerance":                     {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Soil Microbiology":                    {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Fermentation":                         {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Escherichia coli Infections":          {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Candida albicans":                     {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Hepatitis B Surface Antigens":         {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Streptococcal Infections":             {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Antigens Surface":                     {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Malaria":                              {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Immune Adherence Reaction":            {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Hepatitis B Antigens":                 {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Antibody-Producing Cells":             {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Hepatitis":                            {"sub": "Immunology & Microbiology", "weight": 1.0},
    "HIV Infections":                       {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Aerobiosis":                           {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Immunoblotting":                       {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Bacteriophages":                       {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Immunosuppressive Agents":             {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Acquired Immunodeficiency Syndrome":   {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Genes Bacterial":                      {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Food Microbiology":                    {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Hypersensitivity":                     {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Immunoglobulin Fc Fragments":          {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Streptomyces":                         {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Antigens CD":                          {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Antibodies Anti-Idiotypic":            {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Viral Vaccines":                       {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Cytopathogenic Effect Viral":          {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Adjuvants Immunologic":                {"sub": "Immunology & Microbiology", "weight": 1.0},
    "HIV-1":                                {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Interleukin-2":                        {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Chromosomes Bacterial":                {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Schistosomiasis":                      {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Cytomegalovirus":                      {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Bacillus":                             {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Isoantibodies":                        {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Bacterial Vaccines":                   {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Interleukin-1":                        {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Spores Bacterial":                     {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Spores Fungal":                        {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Enterotoxins":                         {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Neisseria gonorrhoeae":                {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Mycobacterium":                        {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Mitosporic Fungi":                     {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Haemophilus influenzae":               {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Immunologic Memory":                   {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Streptococcus pyogenes":               {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Pseudomonas Infections":               {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Immunoglobulin Fab Fragments":         {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Papillomaviridae":                     {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Cytomegalovirus Infections":           {"sub": "Immunology & Microbiology", "weight": 1.0},
    "CD4-Positive T-Lymphocytes":           {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Cytokines":                            {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Immunoassay":                          {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Immunologic Deficiency Syndromes":     {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Herpes Simplex":                       {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Streptococcus pneumoniae":             {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Mycobacterium tuberculosis":           {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Bacterial Toxins":                     {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Virus Cultivation":                    {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Klebsiella pneumoniae":                {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Mycobacterium bovis":                  {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Viral Plaque Assay":                   {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Herpesviridae":                        {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Gonorrhea":                            {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Germ-Free Life":                       {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Hepatitis A":                          {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Candidiasis":                          {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Clostridium":                          {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Lymphokines":                          {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Vibrio cholerae":                      {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Meningitis":                           {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Vesicular stomatitis Indiana virus":   {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Vaccines Attenuated":                  {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Rheumatoid Factor":                    {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Blood Bactericidal Activity":          {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Bacteriuria":                          {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Hepatitis B Antibodies":               {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Immunoglobulin Heavy Chains":          {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Herpesviridae Infections":             {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Tuberculosis":                         {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Vaccinia virus":                       {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Influenza Vaccines":                   {"sub": "Immunology & Microbiology", "weight": 1.0},
    "Interleukin-6":                        {"sub": "Immunology & Microbiology", "weight": 1.0},

    # IMMUNOLOGY & MICROBIOLOGY weight 0.5
    "Leukocyte Count":                      {"sub": "Immunology & Microbiology", "weight": 0.5},
    "Immunosuppression Therapy":            {"sub": "Immunology & Microbiology", "weight": 0.5},
    "Carcinoembryonic Antigen":             {"sub": "Immunology & Microbiology", "weight": 0.5},
    "Disease Outbreaks":                    {"sub": "Immunology & Microbiology", "weight": 0.5},
    "Pneumonia":                            {"sub": "Immunology & Microbiology", "weight": 0.5},
    "Respiratory Tract Infections":         {"sub": "Immunology & Microbiology", "weight": 0.5},
    "Fever":                                {"sub": "Immunology & Microbiology", "weight": 0.5},
    "Cross Infection":                      {"sub": "Immunology & Microbiology", "weight": 0.5},
    "Infections":                           {"sub": "Immunology & Microbiology", "weight": 0.5},
    "Urinary Tract Infections":             {"sub": "Immunology & Microbiology", "weight": 0.5},
    
    # ONCOLOGY weight 1.0
    "Neoplasms Experimental":               {"sub": "Oncology", "weight": 1.0},
    "Neoplasms":                            {"sub": "Oncology", "weight": 1.0},
    "Neoplasm Metastasis":                  {"sub": "Oncology", "weight": 1.0},
    "Breast Neoplasms":                     {"sub": "Oncology", "weight": 1.0},
    "Lung Neoplasms":                       {"sub": "Oncology", "weight": 1.0},
    "Adenocarcinoma":                       {"sub": "Oncology", "weight": 1.0},
    "Liver Neoplasms":                      {"sub": "Oncology", "weight": 1.0},
    "Lymph Nodes":                          {"sub": "Oncology", "weight": 1.0},
    "Carcinoma Squamous Cell":              {"sub": "Oncology", "weight": 1.0},
    "Neoplasm Transplantation":             {"sub": "Oncology", "weight": 1.0},
    "Cell Transformation Neoplastic":       {"sub": "Oncology", "weight": 1.0},
    "Brain Neoplasms":                      {"sub": "Oncology", "weight": 1.0},
    "Lymphoma":                             {"sub": "Oncology", "weight": 1.0},
    "Antigens Neoplasm":                    {"sub": "Oncology", "weight": 1.0},
    "Carcinoma":                            {"sub": "Oncology", "weight": 1.0},
    "Melanoma":                             {"sub": "Oncology", "weight": 1.0},
    "Carcinoma Hepatocellular":             {"sub": "Oncology", "weight": 1.0},
    "Colonic Neoplasms":                    {"sub": "Oncology", "weight": 1.0},
    "DNA Neoplasm":                         {"sub": "Oncology", "weight": 1.0},
    "Skin Neoplasms":                       {"sub": "Oncology", "weight": 1.0},
    "Lymphatic Metastasis":                 {"sub": "Oncology", "weight": 1.0},
    "Neoplasm Recurrence Local":            {"sub": "Oncology", "weight": 1.0},
    "Leukemia":                             {"sub": "Oncology", "weight": 1.0},
    "Stomach Neoplasms":                    {"sub": "Oncology", "weight": 1.0},
    "Leukemia Lymphoid":                    {"sub": "Oncology", "weight": 1.0},
    "Uterine Cervical Neoplasms":           {"sub": "Oncology", "weight": 1.0},
    "Neoplasm Staging":                     {"sub": "Oncology", "weight": 1.0},
    "Leukemia Experimental":                {"sub": "Oncology", "weight": 1.0},
    "Adenoma":                              {"sub": "Oncology", "weight": 1.0},
    "Kidney Neoplasms":                     {"sub": "Oncology", "weight": 1.0},
    "Urinary Bladder Neoplasms":            {"sub": "Oncology", "weight": 1.0},
    "Prostatic Neoplasms":                  {"sub": "Oncology", "weight": 1.0},
    "Bone Neoplasms":                       {"sub": "Oncology", "weight": 1.0},
    "Ovarian Neoplasms":                    {"sub": "Oncology", "weight": 1.0},
    "Pancreatic Neoplasms":                 {"sub": "Oncology", "weight": 1.0},
    "Sarcoma Experimental":                 {"sub": "Oncology", "weight": 1.0},
    "Mammary Neoplasms Experimental":       {"sub": "Oncology", "weight": 1.0},
    "Neoplasm Proteins":                    {"sub": "Oncology", "weight": 1.0},
    "Uterine Neoplasms":                    {"sub": "Oncology", "weight": 1.0},
    "Rectal Neoplasms":                     {"sub": "Oncology", "weight": 1.0},
    "Carcinoma Ehrlich Tumor":              {"sub": "Oncology", "weight": 1.0},
    "Lymphoma Non-Hodgkin":                 {"sub": "Oncology", "weight": 1.0},
    "Leukemia Myeloid Acute":               {"sub": "Oncology", "weight": 1.0},
    "Thyroid Neoplasms":                    {"sub": "Oncology", "weight": 1.0},
    "Leukemia Myeloid":                     {"sub": "Oncology", "weight": 1.0},
    "Glioma":                               {"sub": "Oncology", "weight": 1.0},
    "Antineoplastic Combined Chemotherapy Protocols": {"sub": "Oncology", "weight": 1.0},
    "Head and Neck Neoplasms":              {"sub": "Oncology", "weight": 1.0},
    "RNA Neoplasm":                         {"sub": "Oncology", "weight": 1.0},
    "Neuroblastoma":                        {"sub": "Oncology", "weight": 1.0},
    "Pituitary Neoplasms":                  {"sub": "Oncology", "weight": 1.0},
    "Leukemia L1210":                       {"sub": "Oncology", "weight": 1.0},
    "Multiple Myeloma":                     {"sub": "Oncology", "weight": 1.0},
    "Tumor Virus Infections":               {"sub": "Oncology", "weight": 1.0},
    "Esophageal Neoplasms":                 {"sub": "Oncology", "weight": 1.0},
    "Laryngeal Neoplasms":                  {"sub": "Oncology", "weight": 1.0},
    "Antibodies Neoplasm":                  {"sub": "Oncology", "weight": 1.0},
    "Mouth Neoplasms":                      {"sub": "Oncology", "weight": 1.0},
    "Carcinoma in Situ":                    {"sub": "Oncology", "weight": 1.0},
    "Carcinoma Small Cell":                 {"sub": "Oncology", "weight": 1.0},
    "Fibrosarcoma":                         {"sub": "Oncology", "weight": 1.0},
    "Liver Neoplasms Experimental":         {"sub": "Oncology", "weight": 1.0},
    "Testicular Neoplasms":                 {"sub": "Oncology", "weight": 1.0},
    "Osteosarcoma":                         {"sub": "Oncology", "weight": 1.0},
    "Neoplasm Invasiveness":                {"sub": "Oncology", "weight": 1.0},
    "Sarcoma":                              {"sub": "Oncology", "weight": 1.0},
    "Biomarkers Tumor":                     {"sub": "Oncology", "weight": 1.0},
    "Avian Sarcoma Viruses":                {"sub": "Oncology", "weight": 1.0},
    "Astrocytoma":                          {"sub": "Oncology", "weight": 1.0},
    "Gastrointestinal Neoplasms":           {"sub": "Oncology", "weight": 1.0},
    "Neoplasms Multiple Primary":           {"sub": "Oncology", "weight": 1.0},
    "Carcinoma Transitional Cell":          {"sub": "Oncology", "weight": 1.0},
    "Meningioma":                           {"sub": "Oncology", "weight": 1.0},
    "Lymphoma Large B-Cell Diffuse":        {"sub": "Oncology", "weight": 1.0},
    "Intestinal Neoplasms":                 {"sub": "Oncology", "weight": 1.0},
    "Bronchial Neoplasms":                  {"sub": "Oncology", "weight": 1.0},
    "Adrenal Gland Neoplasms":              {"sub": "Oncology", "weight": 1.0},

    # ONCOLOGY weight 0.5
    "Prognosis":                            {"sub": "Oncology", "weight": 0.5},
    "Biopsy":                               {"sub": "Oncology", "weight": 0.5},
    "Biopsy Needle":                        {"sub": "Oncology", "weight": 0.5},
    "Hyperplasia":                          {"sub": "Oncology", "weight": 0.5},
    "Prostatic Hyperplasia":                {"sub": "Oncology", "weight": 0.5},
    "Epidermal Growth Factor":              {"sub": "Oncology", "weight": 0.5},
    "Cytodiagnosis":                        {"sub": "Oncology", "weight": 0.5},

    # PHARMACOLOGY weight 1.0
    "Norepinephrine":                       {"sub": "Pharmacology", "weight": 1.0},
    "Drug Evaluation":                      {"sub": "Pharmacology", "weight": 1.0},
    "Hydrocortisone":                       {"sub": "Pharmacology", "weight": 1.0},
    "Antineoplastic Agents":                {"sub": "Pharmacology", "weight": 1.0},
    "Isoproterenol":                        {"sub": "Pharmacology", "weight": 1.0},
    "Drug Stability":                       {"sub": "Pharmacology", "weight": 1.0},
    "Histamine":                            {"sub": "Pharmacology", "weight": 1.0},
    "Drug Combinations":                    {"sub": "Pharmacology", "weight": 1.0},
    "Atropine":                             {"sub": "Pharmacology", "weight": 1.0},
    "Indomethacin":                         {"sub": "Pharmacology", "weight": 1.0},
    "Drug Administration Schedule":         {"sub": "Pharmacology", "weight": 1.0},
    "Aspirin":                              {"sub": "Pharmacology", "weight": 1.0},
    "Drug Evaluation Preclinical":          {"sub": "Pharmacology", "weight": 1.0},
    "Triiodothyronine":                     {"sub": "Pharmacology", "weight": 1.0},
    "Dactinomycin":                         {"sub": "Pharmacology", "weight": 1.0},
    "Theophylline":                         {"sub": "Pharmacology", "weight": 1.0},
    "Carcinogens":                          {"sub": "Pharmacology", "weight": 1.0},
    "Lethal Dose 50":                       {"sub": "Pharmacology", "weight": 1.0},
    "Methotrexate":                         {"sub": "Pharmacology", "weight": 1.0},
    "Gentamicins":                          {"sub": "Pharmacology", "weight": 1.0},
    "Chloramphenicol":                      {"sub": "Pharmacology", "weight": 1.0},
    "Doxorubicin":                          {"sub": "Pharmacology", "weight": 1.0},
    "Imidazoles":                           {"sub": "Pharmacology", "weight": 1.0},
    "Penicillins":                          {"sub": "Pharmacology", "weight": 1.0},
    "Adenosine":                            {"sub": "Pharmacology", "weight": 1.0},
    "Anti-Inflammatory Agents":             {"sub": "Pharmacology", "weight": 1.0},
    "Ampicillin":                           {"sub": "Pharmacology", "weight": 1.0},
    "Somatostatin":                         {"sub": "Pharmacology", "weight": 1.0},
    "Vincristine":                          {"sub": "Pharmacology", "weight": 1.0},
    "Analgesics":                           {"sub": "Pharmacology", "weight": 1.0},
    "Fluorouracil":                         {"sub": "Pharmacology", "weight": 1.0},
    "Chlorpromazine":                       {"sub": "Pharmacology", "weight": 1.0},
    "Lidocaine":                            {"sub": "Pharmacology", "weight": 1.0},
    "Haloperidol":                          {"sub": "Pharmacology", "weight": 1.0},
    "Tetracycline":                         {"sub": "Pharmacology", "weight": 1.0},
    "Vitamin A":                            {"sub": "Pharmacology", "weight": 1.0},
    "Pyridines":                            {"sub": "Pharmacology", "weight": 1.0},
    "Phenytoin":                            {"sub": "Pharmacology", "weight": 1.0},
    "Prednisolone":                         {"sub": "Pharmacology", "weight": 1.0},
    "Antihypertensive Agents":              {"sub": "Pharmacology", "weight": 1.0},
    "Anticonvulsants":                      {"sub": "Pharmacology", "weight": 1.0},
    "Oxytocin":                             {"sub": "Pharmacology", "weight": 1.0},
    "Carbachol":                            {"sub": "Pharmacology", "weight": 1.0},
    "Piperazines":                          {"sub": "Pharmacology", "weight": 1.0},
    "Pharmaceutical Preparations":          {"sub": "Pharmacology", "weight": 1.0},
    "Chemical and Drug Induced Liver Injury": {"sub": "Pharmacology", "weight": 1.0},
    "Insecticides":                         {"sub": "Pharmacology", "weight": 1.0},
    "Diuretics":                            {"sub": "Pharmacology", "weight": 1.0},
    "Furosemide":                           {"sub": "Pharmacology", "weight": 1.0},
    "Vasodilator Agents":                   {"sub": "Pharmacology", "weight": 1.0},
    "Cytarabine":                           {"sub": "Pharmacology", "weight": 1.0},
    "Piperidines":                          {"sub": "Pharmacology", "weight": 1.0},
    "Bradykinin":                           {"sub": "Pharmacology", "weight": 1.0},
    "Calcitonin":                           {"sub": "Pharmacology", "weight": 1.0},
    "Propanolamines":                       {"sub": "Pharmacology", "weight": 1.0},
    "Antiviral Agents":                     {"sub": "Pharmacology", "weight": 1.0},
    "Digoxin":                              {"sub": "Pharmacology", "weight": 1.0},
    "Streptomycin":                         {"sub": "Pharmacology", "weight": 1.0},
    "Vitamin E":                            {"sub": "Pharmacology", "weight": 1.0},
    "Calcimycin":                           {"sub": "Pharmacology", "weight": 1.0},
    "Methylcholanthrene":                   {"sub": "Pharmacology", "weight": 1.0},
    "Clonidine":                            {"sub": "Pharmacology", "weight": 1.0},
    "Penicillin Resistance":                {"sub": "Pharmacology", "weight": 1.0},
    "Verapamil":                            {"sub": "Pharmacology", "weight": 1.0},
    "Protease Inhibitors":                  {"sub": "Pharmacology", "weight": 1.0},
    "Vitamin B 12":                         {"sub": "Pharmacology", "weight": 1.0},
    "Nitrosamines":                         {"sub": "Pharmacology", "weight": 1.0},
    "Cisplatin":                            {"sub": "Pharmacology", "weight": 1.0},
    "Antifungal Agents":                    {"sub": "Pharmacology", "weight": 1.0},
    "Phenylephrine":                        {"sub": "Pharmacology", "weight": 1.0},
    "Alkaloids":                            {"sub": "Pharmacology", "weight": 1.0},
    "Bromodeoxyuridine":                    {"sub": "Pharmacology", "weight": 1.0},
    "Penicillin G":                         {"sub": "Pharmacology", "weight": 1.0},
    "Bleomycin":                            {"sub": "Pharmacology", "weight": 1.0},
    "Benzopyrenes":                         {"sub": "Pharmacology", "weight": 1.0},
    "Anti-Inflammatory Agents, Non-Steroidal": {"sub": "Pharmacology", "weight": 1.0},
    "Pyrimidines":                          {"sub": "Pharmacology", "weight": 1.0},
    "Salicylates":                          {"sub": "Pharmacology", "weight": 1.0},
    "Erythromycin":                         {"sub": "Pharmacology", "weight": 1.0},
    "Imipramine":                           {"sub": "Pharmacology", "weight": 1.0},
    "Mitomycins":                           {"sub": "Pharmacology", "weight": 1.0},
    "Vinblastine":                          {"sub": "Pharmacology", "weight": 1.0},
    "Anticoagulants":                       {"sub": "Pharmacology", "weight": 1.0},
    "Kanamycin":                            {"sub": "Pharmacology", "weight": 1.0},
    "Phenoxybenzamine":                     {"sub": "Pharmacology", "weight": 1.0},
    "Nitroglycerin":                        {"sub": "Pharmacology", "weight": 1.0},
    "Methylprednisolone":                   {"sub": "Pharmacology", "weight": 1.0},
    "Diethylstilbestrol":                   {"sub": "Pharmacology", "weight": 1.0},
    "Puromycin":                            {"sub": "Pharmacology", "weight": 1.0},
    "Nitric Oxide":                         {"sub": "Pharmacology", "weight": 1.0},
    "Tretinoin":                            {"sub": "Pharmacology", "weight": 1.0},
    "Bromocriptine":                        {"sub": "Pharmacology", "weight": 1.0},
    "Azathioprine":                         {"sub": "Pharmacology", "weight": 1.0},
    "Histamine H1 Antagonists":             {"sub": "Pharmacology", "weight": 1.0},
    "Antibiotics Antineoplastic":           {"sub": "Pharmacology", "weight": 1.0},
    "Amphotericin B":                       {"sub": "Pharmacology", "weight": 1.0},
    "Arginine Vasopressin":                 {"sub": "Pharmacology", "weight": 1.0},
    "Anti-Infective Agents":                {"sub": "Pharmacology", "weight": 1.0},
    "Aflatoxins":                           {"sub": "Pharmacology", "weight": 1.0},
    "Nitroprusside":                        {"sub": "Pharmacology", "weight": 1.0},
    "Vitamins":                             {"sub": "Pharmacology", "weight": 1.0},
    "Trimethoprim":                         {"sub": "Pharmacology", "weight": 1.0},
    "Benzimidazoles":                       {"sub": "Pharmacology", "weight": 1.0},
    "Dronabinol":                           {"sub": "Pharmacology", "weight": 1.0},
    "Daunorubicin":                         {"sub": "Pharmacology", "weight": 1.0},

    # PHARMACOLOGY weight 0.5
    "Endotoxins":                           {"sub": "Pharmacology", "weight": 0.5},
    "Allergens":                            {"sub": "Pharmacology", "weight": 0.5},
    "Anaphylaxis":                          {"sub": "Pharmacology", "weight": 0.5},
    "Histamine Release":                    {"sub": "Pharmacology", "weight": 0.5},
    "Plants Medicinal":                     {"sub": "Pharmacology", "weight": 0.5},
    "Plant Extracts":                       {"sub": "Pharmacology", "weight": 0.5},
    "Fibrinolysis":                         {"sub": "Pharmacology", "weight": 0.5},
}

In [10]:
# PASS 1: CLASSIFY ALL USABLE PAPERS
# For each PubMed article in all 50 .xml.gz files:
#   - skip if missing title, abstract, or MeSH
#   - skip if publication type is in the exclusion list
#   - score each of our 6 subcategories using weighted MeSH term matching
#   - assign the paper to the subcategory with the highest score
#   - store the paper (title + abstract + subcategory) in a big list

# papers_by_sub is a dictionary where each key is a subcategory name
# (e.g. "Oncology") and the value is a list of paper dictionaries
# Using defaultdict(list) means we don't have to check if a key exists
# before appending, it automatically creates an empty list on first access
papers_by_sub = defaultdict(list)

# Counter for papers that couldn't be classified because none of their
# MeSH terms appear in PUBMED_MESH_MAP
skipped_no_match = 0

# Counter for papers filtered out by publication type
skipped_excluded = 0

# Total number of PubmedArticle elements we attempt to process
total_processed = 0

# List all files in the pubmed_raw folder, sorted alphabetically
# This ensures consistent order across runs
files = sorted(os.listdir(PUBMED_DIR))

# LOOP OVER EACH FILE
for filename in files:
    file_path = os.path.join(PUBMED_DIR, filename)

    try:
        # Open the .gz file
        with gzip.open(file_path, 'rb') as f:

            # iterparse streams the XML, firing an 'end' event each time
            # a closing tag is read. We only care about full PubmedArticle elements
            for event, elem in ET.iterparse(f, events=('end',)):
                if elem.tag != 'PubmedArticle':
                    continue  # skip intermediate elements (Journal, MeshHeading, etc.)

                total_processed += 1

                # 1. Title
                # findtext returns the text of the first matching sub-element
                # We check that it's not None and not empty after stripping
                title = elem.findtext('.//ArticleTitle')
                if not title or not title.strip():
                    elem.clear()  # free memory, skip to next article
                    continue

                # 2. Abstract
                # PubMed abstracts can be simple or structured
                # We join all parts with a space, which works for both formats
                abstract_parts = elem.findall('.//AbstractText')
                abstract = ' '.join(
                    (p.text or '') for p in abstract_parts
                ).strip()

                if not abstract:
                    elem.clear()
                    continue

                # 3. MeSH Headings
                mesh_headings = elem.findall('.//MeshHeading')
                if not mesh_headings:  # no MeSH terms → unusable
                    elem.clear()
                    continue

                # 4. Publication Type Filter
                # Collect all publication types for this article into a set
                pub_types = set()
                for pt in elem.findall('.//PublicationType'):
                    if pt.text:
                        pub_types.add(pt.text)

                # If any of its types are in our exclusion list, skip the paper
                # The & operator is set intersection — it returns the common elements
                if pub_types & EXCLUDE_PUB_TYPES:
                    skipped_excluded += 1
                    elem.clear()
                    continue

                # 5. Score Subcategories
                # We'll accumulate a float score for each subcategory
                scores = defaultdict(float)

                # For every MeSH heading on the paper
                for mh in mesh_headings:
                    descriptor = mh.find('DescriptorName')
                    if descriptor is not None and descriptor.text:
                        name = descriptor.text.strip()

                        # Look up this term in our manually built map
                        if name in PUBMED_MESH_MAP:
                            entry = PUBMED_MESH_MAP[name]
                            # Add the term's weight to the appropriate subcategory
                            # weight = 1.0 for specific terms, 0.5 for ambiguous ones
                            scores[entry['sub']] += entry['weight']
                        # If the term isn't in the map, we simply ignore it

                # If no term matched the map at all, we can't classify the paper
                if not scores:
                    skipped_no_match += 1
                    elem.clear()
                    continue

                # 6. Determine Winner
                # Find the highest score among all subcategories
                max_score = max(scores.values())

                # Get all subcategories that achieved this top score
                candidates = [sub for sub, sc in scores.items() if sc == max_score]

                if len(candidates) == 1:
                    # Only one subcategory at the top → clear winner
                    winner = candidates[0]
                else:
                    # Tiebreaker: use the predefined priority order
                    # Lower index = higher priority. We iterate through the
                    # priority list and pick the first candidate that appears
                    for sub in PRIORITY_ORDER:
                        if sub in candidates:
                            winner = sub
                            break
                    else:
                        # Fallback:
                        # pick the first candidate alphabetically or by hash
                        winner = candidates[0]

                # 7. Store the Paper
                papers_by_sub[winner].append({
                    'title': title.strip(),
                    'abstract': abstract,
                    'main_category': 'Biology & Medicine',  # fixed for the whole dataset
                    'subcategory': winner
                })

                # Free memory by clearing the element and all its children
                # Without this, 1.5 million articles would accumulate in RAM
                elem.clear()

    except (EOFError, ET.ParseError) as e:
        # Handle corrupted .gz files
        # We skip them entirely and report the error
        print(f"  Skipped file {filename}: {e}")

# REPORT
print("Pass 1 complete.")
print(f"Total articles scanned:         {total_processed:>8,}")
print(f"Excluded by publication type:   {skipped_excluded:>8,}")
print(f"No matching MeSH terms:         {skipped_no_match:>8,}")
print("Papers per subcategory:")
for sub in PRIORITY_ORDER:
    print(f"  {sub}: {len(papers_by_sub[sub]):>8,}")

  Skipped file pubmed26n0045.xml.gz: Compressed file ended before the end-of-stream marker was reached
  Skipped file pubmed26n0046.xml.gz: Compressed file ended before the end-of-stream marker was reached
Pass 1 complete.
Total articles scanned:         1,467,775
Excluded by publication type:     69,637
No matching MeSH terms:          220,512
Papers per subcategory:
  Oncology:   48,395
  Neuroscience:   56,955
  Genetics & Molecular Biology:   85,264
  Immunology & Microbiology:   94,923
  Cardiology:   63,082
  Pharmacology:   47,178


In [11]:
# PASS 2: SAMPLE AND SAVE THE FINAL DATASET
# Now that all papers are grouped by subcategory in papers_by_sub,
# we sample up to CAP_PER_SUB (20,000) papers from each group.
# Then we deduplicate, shuffle, and save to dataset_pubmed.csv.

# Set the random seed so sampling is reproducible
random.seed(42)

# This list will hold all sampled paper dictionaries
all_sampled = []

# SAMPLE FROM EACH SUBCATEGORY
for sub in PRIORITY_ORDER:
    papers = papers_by_sub[sub]

    if len(papers) > CAP_PER_SUB:
        # We have more than enough; randomly pick exactly CAP_PER_SUB papers
        sampled = random.sample(papers, CAP_PER_SUB)
    else:
        # Shortfall: take all available papers and warn the user
        print(f"WARNING: {sub} only has {len(papers)} papers (below {CAP_PER_SUB})")
        sampled = papers

    # Double-check each paper's subcategory field is set correctly
    for p in sampled:
        p['subcategory'] = sub

    # Add this subcategory's sampled papers to the master list
    all_sampled.extend(sampled)

df = pd.DataFrame(all_sampled)

# DEDUPLICATE
# In rare cases, the same paper might appear in multiple PubMed files
# We drop rows that have both the same title AND the same abstract
df = df.drop_duplicates(subset=['title', 'abstract'])

# SHUFFLE
# We don't want the model to see all Oncology papers first, then all
# Cardiology papers. Shuffling randomizes the order
# frac=1 means "take 100% of the rows", which effectively shuffles the whole DataFrame
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# SAVE TO CSV
# The index=False means we don't save the row numbers as an extra column
df.to_csv(PUBMED_OUTPUT, index=False)
print(f"\nFinal dataset saved to {PUBMED_OUTPUT}")
print(f"Total papers: {len(df):,}")

# QUICK DISTRIBUTION CHECK
print("\nPapers per subcategory:")
print(df['subcategory'].value_counts())


Final dataset saved to C:\Users\hp\Desktop\Research-System\training\datasets\dataset_pubmed.csv
Total papers: 119,997

Papers per subcategory:
subcategory
Genetics & Molecular Biology    20000
Neuroscience                    20000
Immunology & Microbiology       20000
Cardiology                      20000
Pharmacology                    19999
Oncology                        19998
Name: count, dtype: int64


In [12]:
# RENAME COLUMNS TO MATCH ARXIV FORMAT
df = df.rename(columns={
    'main_category': 'main_label',
    'subcategory': 'sub_label'
})
df['source'] = 'pubmed'

# overwrite the CSV with the corrected column names
df.to_csv(PUBMED_OUTPUT, index=False)
print("Columns updated and CSV overwritten.")
print(f"Columns now: {list(df.columns)}")

Columns updated and CSV overwritten.
Columns now: ['title', 'abstract', 'main_label', 'sub_label', 'source']


In [13]:
# CSV INSPECTION
# Load the saved CSV and run a full quality check

df = pd.read_csv(PUBMED_OUTPUT)

print("=" * 60)
print("SHAPE & COLUMNS")
print("=" * 60)
print(f"Rows: {len(df):,}")
print(f"Columns: {list(df.columns)}")

print("\n" + "=" * 60)
print("DISTRIBUTION BY SUBCATEGORY")
print("=" * 60)
print(df['sub_label'].value_counts().to_string())

print("\n" + "=" * 60)
print("DISTRIBUTION BY MAIN CATEGORY")
print("=" * 60)
print(df['main_label'].value_counts().to_string())

print("\n" + "=" * 60)
print("MISSING VALUES")
print("=" * 60)
print(df.isnull().sum())

print("\n" + "=" * 60)
print("EMPTY STRINGS")
print("=" * 60)
print((df == '').sum())

print("\n" + "=" * 60)
print("DUPLICATES")
print("=" * 60)
exact_dupes = df.duplicated(subset=['title', 'abstract']).sum()
title_dupes = df.duplicated(subset=['title']).sum()
print(f"Exact duplicates (title + abstract): {exact_dupes:,}")
print(f"Title-only duplicates:               {title_dupes:,}")

print("\n" + "=" * 60)
print("ABSTRACT LENGTH STATS (characters)")
print("=" * 60)
df['abstract_len'] = df['abstract'].str.len()
print(df['abstract_len'].describe().round(1).to_string())
very_short = (df['abstract_len'] < 100).sum()
very_long  = (df['abstract_len'] > 3000).sum()
print(f"\nAbstracts under 100 chars:  {very_short:,}")
print(f"Abstracts over 3000 chars:  {very_long:,}")

print("\n" + "=" * 60)
print("TITLE LENGTH STATS (characters)")
print("=" * 60)
df['title_len'] = df['title'].str.len()
print(df['title_len'].describe().round(1).to_string())
empty_titles = (df['title_len'] < 5).sum()
print(f"\nTitles under 5 chars: {empty_titles:,}")

print("\n" + "=" * 60)
print("LEFTOVER LATEX CHECK (sample of suspicious rows)")
print("=" * 60)
latex_mask = df['abstract'].str.contains(r'\\[a-zA-Z]|\$', regex=True, na=False)
print(f"Abstracts with possible leftover LaTeX: {latex_mask.sum():,}")
if latex_mask.sum() > 0:
    print("\nSample abstracts with LaTeX:")
    sample = df[latex_mask]['abstract'].head(3).tolist()
    for i, s in enumerate(sample):
        print(f"\n  [{i+1}] {s[:300]}")

print("\n" + "=" * 60)
print("SOURCE COLUMN CHECK")
print("=" * 60)
print(df['source'].value_counts().to_string())

print("\n" + "=" * 60)
print("5 RANDOM SAMPLE ROWS")
print("=" * 60)
sample_df = df.sample(5, random_state=42)[['title', 'abstract', 'main_label', 'sub_label']]
for _, row in sample_df.iterrows():
    print(f"\n  Title:    {row['title'][:100]}")
    print(f"  Abstract: {row['abstract'][:200]}")
    print(f"  Label:    {row['main_label']} › {row['sub_label']}")
    print()

SHAPE & COLUMNS
Rows: 119,997
Columns: ['title', 'abstract', 'main_label', 'sub_label', 'source']

DISTRIBUTION BY SUBCATEGORY
sub_label
Genetics & Molecular Biology    20000
Neuroscience                    20000
Immunology & Microbiology       20000
Cardiology                      20000
Pharmacology                    19999
Oncology                        19998

DISTRIBUTION BY MAIN CATEGORY
main_label
Biology & Medicine    119997

MISSING VALUES
title         0
abstract      0
main_label    0
sub_label     0
source        0
dtype: int64

EMPTY STRINGS
title         0
abstract      0
main_label    0
sub_label     0
source        0
dtype: int64

DUPLICATES
Exact duplicates (title + abstract): 0
Title-only duplicates:               45

ABSTRACT LENGTH STATS (characters)
count    119997.0
mean        994.9
std         472.3
min          73.0
25%         643.0
50%         929.0
75%        1268.0
max        4279.0

Abstracts under 100 chars:  17
Abstracts over 3000 chars:  135

TITLE LENGT